---
# **Peta  Kebakarang Hutang di Indonesia**

**NAMA  : CAROL DUPINO PEREIRA**

**NIM   : 52250051**

**PROGRAM STUDI : SAINS DATA**

**MATA KULIAH   : ANALISI DATA ESPASIAL** 

---

Versi dengan satu sel khusus untuk setiap provinsi. Jalankan **Run All**, lalu lihat **bagian 9: Analisis masing-masing provinsi**. Setiap provinsi menampilkan peta titik panas, nama kabupaten/kota, tabel jumlah deteksi, dan koordinat titik.

Notebook harus berada bersama skrip dan data sumber. Titik panas bukan konfirmasi kebakaran hutan; periode luas dan deteksi berbeda.

## Sumber dataset dan ruang lingkup analisis

| Dataset | Berkas dan cakupan | URL sumber |
| --- | --- | --- |
| NASA FIRMS VIIRS NOAA-20 NRT | `All Dataset/firms_noaa20_indonesia_24h.csv`, titik berkoordinat dalam batas Indonesia, snapshot 24 jam | [Unduhan CSV resmi](https://firms.modaps.eosdis.nasa.gov/data/active_fire/noaa-20-viirs-c2/csv/J1_VIIRS_C2_SouthEast_Asia_24h.csv) dan [portal NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/active_fire/) |
| Luas karhutla SIPONGI KEMENHUT | `All Dataset/luas-karhutla-provinsi.xlsx`, kolom 2018–2026, ha | [Indikasi luas kebakaran](https://sipongi.gakkum.kehutanan.go.id/indikasi-luas-kebakaran) |
| Batas administrasi | `All Dataset/provinsi.geojson` dan `kab_kota.geojson`, 38 provinsi dan 514 kabupaten/kota | [Indonesia-GeoJSON](https://github.com/AlfianAliM/Indonesia-GeoJSON), [Laravel Nusa](https://nusa.creasi.dev), [Peta Nusa](https://petanusa.web.id) |
| Tujuh folder statistik kebakaran | Kehilangan tutupan pohon 2001–2025 dan statistik peringatan VIIRS, metadata unduhan 10 November 2026 | [Global Nature Watch — Indonesia, kategori fires](https://globalnaturewatch.org/dashboards/country/IDN/?category=fires&lang=id&map=eyJjYW5Cb3VuZCI6dHJ1ZX0%3D) |

`unduh_firms.py` menyimpan CSV regional asli, subset Indonesia, URL unduhan, tanggal pengunduhan UTC, periode pengamatan, dan aturan seleksi pada JSON metadata. Titik diseleksi menggunakan batas provinsi Indonesia yang diperbaiki, sehingga lokasi di luar poligon sumber dikeluarkan. Nama confidence dinormalisasi menjadi l/n/h. Kolom `brightness` dan `bright_t31` pada subset mempertahankan nama lama proyek dan berasal dari `bright_ti4` serta `bright_ti5` VIIRS. Nilai asal tersedia dalam CSV regional.

Titik panas merupakan indikasi kebakaran atau anomali panas satelit. Identifikasi kebakaran pada tutupan hutan memerlukan data tutupan lahan dan pemeriksaan pendukung. Confidence menggambarkan kualitas deteksi satelit. Data kehilangan tutupan pohon akibat kebakaran merupakan estimasi dampak tahunan dengan metode sumber UMD/GLAD. [Metode kehilangan tutupan pohon WRI](https://gfr.wri.org/annual-tree-cover-loss-data-explained)

## Sistem koordinat dan metode choropleth

Analisis menggunakan **WGS 84 / EPSG:4326**. Longitude ditempatkan pada sumbu X dan latitude pada sumbu Y. CRS ini sesuai dengan koordinat CSV dan mendukung tampilan seluruh Indonesia. Spatial join dilakukan setelah CRS titik dan poligon disamakan. Panah U menunjukkan utara geografis pada peta bujur–lintang ini.

Choropleth nasional memakai jumlah deteksi yang cocok per provinsi dengan gradasi YlOrBr. Choropleth kabupaten memakai jumlah deteksi dalam provinsi terpilih. Peta luas SIPONGI dan kehilangan tutupan pohon GNW memakai ha dengan gradasi YlOrRd. Colorbar menunjukkan skala numerik. Nilai kosong pada GNW dipertahankan sebagai data tidak tersedia. Jumlah deteksi adalah ukuran absolut, sehingga perbandingan dipengaruhi luas wilayah dan cakupan pengamatan satelit.


## **1. Persiapan**

Pilih kernel Python yang memiliki paket pada `requirements.txt`. Jika diperlukan, jalankan `%pip install -r requirements.txt` pada sel baru, lalu restart kernel.

In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import geopandas as gpd
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
DATA_DIR = ROOT / 'All Dataset'
required = [ROOT / 'peta_karhutla.py', ROOT / 'analisis_provinsi.py', ROOT / 'analisis_gnw.py', ROOT / 'elemen_peta.py'] + [
    DATA_DIR / name for name in ['provinsi.geojson', 'kab_kota.geojson',
    'luas-karhutla-provinsi.xlsx', 'firms_noaa20_indonesia_24h.csv']
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Buka notebook dari folder proyek. File tidak ditemukan: {missing}')
print('Folder proyek:', ROOT)
print('Folder dataset:', DATA_DIR)
print('GeoPandas:', gpd.__version__)


## **2. Parameter analisis**

`TAHUN` memilih kolom luas SIPONGI. `SATUAN` menggunakan hektare (ha). CSV bawaan adalah snapshot NOAA-20 24 jam yang diunduh dari NASA FIRMS. Untuk memperbaruinya, jalankan `python unduh_firms.py` dari terminal lalu Run All. Tanggal pengamatan dibaca dari isi CSV dan waktu pengamatan memakai UTC.

In [ ]:
TAHUN = 2026
SATUAN = 'ha'
CSV = DATA_DIR / 'firms_noaa20_indonesia_24h.csv'

## **3. Periksa data sumber**

Koordinat longitude menjadi sumbu X, latitude menjadi sumbu Y. Kode wilayah dipakai untuk menggabungkan Excel dengan batas provinsi.

In [ ]:
titik = pd.read_csv(CSV, dtype={'acq_time': str})
luas = pd.read_excel(DATA_DIR / 'luas-karhutla-provinsi.xlsx', sheet_name='data')
provinsi = gpd.read_file(DATA_DIR / 'provinsi.geojson')
print(f'{len(titik):,} deteksi; {len(provinsi)} provinsi; CRS: {provinsi.crs}')
print('Tanggal deteksi:', sorted(titik.acq_date.unique()))
print('Geometri awal tidak valid:', int((~provinsi.is_valid).sum()))
display(titik.head())
display(luas.head())

# Koordinat sumber longitude/latitude menggunakan WGS 84.
CRS = 'EPSG:4326'
titik_geometri = gpd.GeoDataFrame(titik.copy(), geometry=gpd.points_from_xy(titik.longitude, titik.latitude), crs=CRS)
provinsi = provinsi.to_crs(CRS)
assert titik_geometri.crs == provinsi.crs
print('CRS analisis dan peta:', CRS)
print('Rentang tanggal pengamatan:', titik.acq_date.min(), 'sampai', titik.acq_date.max())
display(titik_geometri[['latitude', 'longitude', 'geometry']].head())
metadata_firms = json.loads((DATA_DIR / 'firms_noaa20_indonesia_24h_metadata.json').read_text(encoding='utf-8'))
display(pd.Series(metadata_firms, name='Snapshot NASA').to_frame())

## **4. Jalankan pengolahan GeoPandas**

Sel ini menjalankan `peta_karhutla.py` dengan kernel Python yang sama. Implementasi pengolahan tetap berada dalam satu skrip agar notebook dan hasil terminal konsisten.

Alurnya: perbaiki geometri dengan `make_valid`, gabungkan bagian poligon per kode, konversi angka Indonesia pada Excel, gabungkan luas berdasarkan kode wilayah, kemudian lakukan spatial join titik dengan provinsi memakai `intersects`. Titik yang cocok dengan beberapa provinsi ditandai ambigu. Titik di luar poligon tetap disimpan.

Menjalankan ulang sel ini memperbarui hasil pada folder `outputs`. Data sumber tetap utuh.

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'peta_karhutla.py'), '--tahun', str(TAHUN), '--satuan', SATUAN, '--csv', str(CSV)],
    cwd=ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace'
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

## **5. Validasi hasil**

Jumlah deteksi harus terjaga setelah pencocokan. Jumlah nilai luas provinsi dibandingkan dengan baris Total Excel menggunakan toleransi desimal.

In [ ]:
OUT = ROOT / 'outputs'
validasi = json.loads((OUT / 'validasi.json').read_text(encoding='utf-8'))
ringkasan = pd.read_csv(OUT / 'ringkasan_provinsi.csv', dtype={'code': str})
hasil_titik = pd.read_csv(OUT / 'titik_panas_dengan_provinsi.csv', dtype={'code': str})
assert len(hasil_titik) == validasi['koordinat_valid']
assert sum(validasi['status_wilayah'].values()) == validasi['koordinat_valid']
assert ringkasan.jumlah_deteksi.sum() == validasi['status_wilayah'].get('cocok', 0)
if validasi['total_luas_excel'] is not None:
    assert abs(validasi['jumlah_luas_provinsi'] - validasi['total_luas_excel']) < 0.01
display(pd.Series(validasi, name='Hasil validasi').to_frame())
display(ringkasan[['code', 'name', 'luas_karhutla', 'jumlah_deteksi']].sort_values('jumlah_deteksi', ascending=False).head(10))

## **6. Tampilkan peta**

Panel atas menampilkan choropleth jumlah deteksi per provinsi dan lokasi pengamatan VIIRS. Warna titik menunjukkan confidence. Panel bawah menampilkan luas karhutla SIPONGI sesuai kolom tahun Excel. Setiap panel dilengkapi panah utara, legenda, judul, dan label EPSG:4326.

In [ ]:
display(Image(filename=str(OUT / 'peta_karhutla.png')))

## **7. Periksa titik yang belum cocok**

Lokasi di luar poligon dapat berhubungan dengan batas pesisir, ketelitian data, atau lokasi di luar wilayah. Status tersebut perlu diperiksa sebelum menetapkan penyebabnya; titik tidak otomatis dipindahkan ke provinsi terdekat.

In [ ]:
belum_cocok = hasil_titik[hasil_titik.status_wilayah.ne('cocok')]
display(belum_cocok[['id_deteksi', 'latitude', 'longitude', 'acq_date', 'confidence', 'status_wilayah']])

## Hasil dan interpretasi

Peta PNG, ringkasan CSV, GeoJSON, dan validasi tersimpan di `outputs/`. Data kabupaten/kota digunakan pada bagian 9. Satuan luas SIPONGI adalah ha, sedangkan tanggal pemutakhiran Excel lokal perlu dicocokkan dengan versi unduhannya. Titik panas dan luas karhutla mengikuti periode masing-masing. Analisis kehilangan tutupan pohon akibat kebakaran tersedia pada bagian 10.

Referensi metode [make_valid](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.make_valid.html) dan [spatial join](https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html).

## **8. Warna wilayah dan peringkat provinsi**

Setiap provinsi diberi warna berbeda untuk mengenali wilayah. Grafik berikutnya mengurutkan seluruh provinsi berdasarkan luas karhutla dan jumlah deteksi secara terpisah. Nilai terbesar menunjukkan dampak absolut tertinggi pada masing-masing ukuran, bukan persentase wilayah terbakar.

In [ ]:
display(Image(filename=str(OUT / 'peta_warna_provinsi.png')))
display(Image(filename=str(OUT / 'peringkat_provinsi.png')))
peringkat = pd.read_csv(OUT / 'peringkat_provinsi.csv', dtype={'code': str})
display(peringkat)

## **9. Analisis masing-masing provinsi**

Jalankan persiapan ini sekali, lalu pilih sel provinsi atau jalankan semuanya. Titik dipetakan ke kabupaten/kota dengan spatial join; hasil ambigu dan tidak cocok tetap ditandai. Nomor pada peta merujuk nama kabupaten/kota di legenda. Tabel lokasi mencantumkan koordinat tiap deteksi.

Jumlah deteksi digunakan untuk membandingkan kabupaten dalam tanggal CSV. Nilai luas hanya tersedia pada tingkat provinsi, bukan kabupaten.

In [ ]:
from analisis_provinsi import siapkan_data, tampilkan_provinsi
data_provinsi = siapkan_data(ROOT)
display(data_provinsi['titik'].status_kabupaten.value_counts().rename_axis('status').to_frame('jumlah_deteksi'))

### Aceh

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Aceh')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sumatera Utara

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sumatera Utara')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sumatera Barat

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sumatera Barat')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Riau

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Riau')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Jambi

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Jambi')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sumatera Selatan

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sumatera Selatan')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Bengkulu

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Bengkulu')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Lampung

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Lampung')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kepulauan Bangka Belitung

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kepulauan Bangka Belitung')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kepulauan Riau

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kepulauan Riau')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Daerah Khusus Ibukota Jakarta

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Daerah Khusus Ibukota Jakarta')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Jawa Barat

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Jawa Barat')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Jawa Tengah

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Jawa Tengah')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Daerah Istimewa Yogyakarta

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Daerah Istimewa Yogyakarta')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Jawa Timur

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Jawa Timur')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Banten

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Banten')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Bali

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Bali')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Nusa Tenggara Barat

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Nusa Tenggara Barat')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Nusa Tenggara Timur

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Nusa Tenggara Timur')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kalimantan Barat

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kalimantan Barat')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kalimantan Tengah

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kalimantan Tengah')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kalimantan Selatan

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kalimantan Selatan')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kalimantan Timur

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kalimantan Timur')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Kalimantan Utara

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Kalimantan Utara')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sulawesi Utara

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sulawesi Utara')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sulawesi Tengah

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sulawesi Tengah')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sulawesi Selatan

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sulawesi Selatan')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sulawesi Tenggara

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sulawesi Tenggara')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Gorontalo

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Gorontalo')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Sulawesi Barat

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Sulawesi Barat')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Maluku

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Maluku')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Maluku Utara

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Maluku Utara')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Papua

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Papua')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Papua Barat

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Papua Barat')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Papua Selatan

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Papua Selatan')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Papua Tengah

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Papua Tengah')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Papua Pegunungan

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Papua Pegunungan')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

### Papua Barat Daya

In [ ]:
hasil = tampilkan_provinsi(data_provinsi, 'Papua Barat Daya')
display(Image(filename=str(hasil['peta'])))
display(hasil['informasi'])
display(hasil['ringkasan'])
display(hasil['titik'])

## **10. Kehilangan tutupan pohon akibat kebakaran — Global Nature Watch**

Analisis memakai folder `Wilayah dengan kehilangan tutupan pohon terbanyak akibat kebakaran di Indonesia` dan kolom `umd_tree_cover_loss_from_fires__ha`. Tahun terakhir diambil langsung dari CSV. Metadata lokal menyatakan ambang tutupan tajuk 30%.

Tabel memakai 34 wilayah. Untuk visualisasi, Papua Selatan, Papua Tengah, dan Papua Pegunungan digabung dengan Papua, sedangkan Papua Barat Daya digabung dengan Papua Barat. Ini merupakan pendekatan penyelarasan wilayah menggunakan batas yang tersedia. Batas sumber historis dapat berbeda. ID `adm1` GNW dihubungkan melalui `adm1_metadata.csv` karena sistem kodenya berbeda dari kode Kemendagri.

In [ ]:
from analisis_gnw import analisis_kehilangan_pohon
hasil_gnw = analisis_kehilangan_pohon(ROOT)
print('Tahun terakhir data kehilangan tutupan pohon:', hasil_gnw['tahun'])
display(Image(filename=str(hasil_gnw['peta'])))
display(hasil_gnw['tabel'].sort_values('kehilangan_akibat_api_ha', ascending=False))

## **11. Inventaris tujuh folder dan pemeriksaan hasil**

Inventaris berikut membaca sumber dan parameter langsung dari `metadata.csv`. Dua folder grafik kehilangan nasional memakai tabel yang sama, dan dua folder peringatan kumulatif/mingguan juga memakai tabel yang sama. Periode parameter dashboard dapat berbeda dari rentang baris CSV ekspor, sehingga filter tahun perlu mengikuti kolom data. Statistik ini dihubungkan ke poligon melalui nama wilayah untuk visualisasi.

In [ ]:
inventaris = []
for folder in sorted(DATA_DIR.iterdir()):
    metadata_path = folder / 'metadata.csv'
    if folder.is_dir() and metadata_path.exists():
        m = pd.read_csv(metadata_path).set_index('title').iloc[:, 0].to_dict()
        inventaris.append({'folder': folder.name, 'tanggal_unduh': m.get('date_downloaded'), 'awal': m.get('start_year', m.get('start_date_absolute', m.get('min_date'))), 'akhir': m.get('end_year', m.get('end_date_absolute', m.get('end_date'))), 'url': m.get('link'), 'metadata_url': m.get('metadata')})
display(pd.DataFrame(inventaris))
assert len(inventaris) == 7
assert len(hasil_gnw['tabel']) == 34
assert int(ringkasan.jumlah_deteksi.sum()) == validasi['status_wilayah'].get('cocok', 0)
print('Periode FIRMS:', validasi['periode_titik_panas'])
print('Deteksi cocok provinsi:', validasi['status_wilayah'].get('cocok', 0))
print('Tahun luas SIPONGI:', TAHUN)
print('Tahun kehilangan tutupan pohon GNW:', hasil_gnw['tahun'])
print('Hasil PNG nasional, GNW, dan 38 provinsi tersedia di outputs/.')